# 📦 Challenge 14: 대용량 데이터 처리

**난이도:** ⭐⭐⭐

**선수 과제:**
- 04_kafka_batch_operations.ipynb
- 05_redis_caching.ipynb

**네비게이션:**
- [← 이전: Challenge 13](13_challenge_distributed_tracing.ipynb)
- [→ 다음: Challenge 15](15_challenge_advanced.ipynb)

## 시나리오: 쿠팡 일일 주문 처리

쿠팡과 같은 대규모 전자상거래 플랫폼에서는 하루에 수백만 건의 주문이 발생합니다.

**문제 상황:**
- 1000건의 주문 데이터를 처리해야 함
- 개별 전송 vs 배치 전송의 성능 차이는?
- 캐싱 전략에 따른 성능 차이는?

**목표:**
1. Kafka 개별 전송 vs 배치 전송 비교
2. Redis 개별 캐싱 vs Pipeline 비교
3. 성능 벤치마크 및 최적화 전략 도출

## 아키텍처

```
┌─────────────────┐
│  1000 Orders    │
│  (bulk_orders)  │
└────────┬────────┘
         │
    ┌────┴────┐
    │         │
    ▼         ▼
┌─────────┐ ┌─────────────┐
│Individual│ │Batch Send   │
│Send     │ │(100 × 10)   │
│1000 RTT │ │10 RTT       │
└────┬────┘ └──────┬──────┘
     │             │
     └──────┬──────┘
            ▼
    ┌──────────────┐
    │ Kafka Broker │
    └──────────────┘

┌─────────────────┐
│  1000 Orders    │
└────────┬────────┘
         │
    ┌────┴────┐
    │         │
    ▼         ▼
┌─────────┐ ┌─────────────┐
│Individual│ │Pipeline     │
│Cache    │ │(Batch Cache)│
│1000 RTT │ │1 RTT        │
└────┬────┘ └──────┬──────┘
     │             │
     └──────┬──────┘
            ▼
    ┌──────────────┐
    │ Redis Cache  │
    └──────────────┘
```

## 벤치마킹 패턴

**성능 측정 방법론:**

1. **네트워크 RTT (Round Trip Time):**
   - 개별 전송: RTT × N (N = 메시지 수)
   - 배치 전송: RTT × (N / batch_size)

2. **측정 지표:**
   - 총 처리 시간 (초)
   - 처리량 (messages/sec)
   - 속도 향상 배수 (Speedup Ratio)

3. **테스트 조건:**
   - 동일한 1000건 데이터셋 사용
   - 배치 크기: 100건
   - 각 테스트 전 캐시 클리어

In [ ]:
# Setup - 필요한 라이브러리 및 데이터 로드
import httpx
import asyncio
import json
import time
from pathlib import Path
from typing import List, Dict, Any

# API 엔드포인트
BASE_URL = "http://localhost:8000"

# 벤치마크 데이터 로드
data_path = Path("../data/mock/bulk_orders.json")

# 데이터 파일이 없으면 생성
if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    # 1000건의 모의 주문 데이터 생성
    bulk_orders = [
        {
            "order_id": f"ORD{i:06d}",
            "user_id": f"USER{(i % 100):04d}",
            "product_id": f"PROD{(i % 500):05d}",
            "quantity": (i % 5) + 1,
            "price": round(10000 + (i * 123.45) % 90000, 2),
            "status": ["pending", "confirmed", "processing"][i % 3],
            "timestamp": f"2026-02-{(i % 28) + 1:02d}T{(i % 24):02d}:{(i % 60):02d}:00Z"
        }
        for i in range(1, 1001)
    ]
    
    with open(data_path, 'w', encoding='utf-8') as f:
        json.dump(bulk_orders, f, ensure_ascii=False, indent=2)
    print(f"✅ 생성됨: {data_path}")
else:
    print(f"✅ 로드됨: {data_path}")

# 데이터 로드
with open(data_path, 'r', encoding='utf-8') as f:
    orders = json.load(f)

print(f"\n📊 데이터셋 메타데이터:")
print(f"   총 주문 수: {len(orders):,}건")
print(f"   데이터 크기: {data_path.stat().st_size / 1024:.2f} KB")
print(f"\n📦 샘플 주문 (처음 3건):")
for order in orders[:3]:
    print(f"   {order['order_id']}: {order['product_id']} × {order['quantity']} = ₩{order['price']:,.0f}")

## Step 1: Kafka 개별 전송 (Individual Send)

**왜 느린가?**
- 각 주문마다 HTTP 요청 1회 = 네트워크 RTT × 1000
- 각 요청마다 TCP 연결/해제 오버헤드
- Kafka 프로듀서가 매번 별도로 메시지 전송
- I/O 대기 시간 누적

**예상 성능:**
- RTT 10ms 가정 시: 10ms × 1000 = 10초 이상
- 처리량: ~100 messages/sec

In [ ]:
# Step 1: 개별 전송 벤치마크
async def send_individual_kafka(orders: List[Dict[str, Any]]) -> float:
    """1000건을 개별적으로 Kafka에 전송"""
    start_time = time.time()
    
    async with httpx.AsyncClient(timeout=60.0) as client:
        tasks = []
        for order in orders:
            task = client.post(
                f"{BASE_URL}/kafka/basic/publish",
                json={
                    "content": json.dumps(order),
                    "metadata": {"topic": "bulk-orders"}
                }
            )
            tasks.append(task)
        
        # 모든 요청 동시 실행 (하지만 각각이 별도 HTTP 요청)
        responses = await asyncio.gather(*tasks)
        
        # 성공 확인
        success_count = sum(1 for r in responses if r.status_code == 200)
    
    elapsed = time.time() - start_time
    
    print(f"⏱️  개별 전송 결과:")
    print(f"   총 처리 시간: {elapsed:.2f}초")
    print(f"   성공: {success_count}/{len(orders)}건")
    print(f"   처리량: {len(orders)/elapsed:.2f} messages/sec")
    
    return elapsed

# 실행
print("🚀 개별 전송 시작 (1000건)...\n")
individual_time = await send_individual_kafka(orders)
print(f"\n✅ 완료!")

## Step 2: Kafka 배치 전송 (Batch Send)

**왜 빠른가?**
- 100건씩 묶어서 전송 = 네트워크 RTT × 10
- HTTP 연결 재사용 (Keep-Alive)
- Kafka 프로듀서의 내부 배치 최적화 활용
- 네트워크 대역폭 효율적 활용

**예상 성능:**
- RTT 10ms 가정 시: 10ms × 10 = 100ms 정도
- 처리량: ~10,000 messages/sec
- 속도 향상: **약 100배**

In [ ]:
# Step 2: 배치 전송 벤치마크
async def send_batch_kafka(orders: List[Dict[str, Any]], batch_size: int = 100) -> float:
    """1000건을 배치로 Kafka에 전송 (100개씩 × 10회)"""
    start_time = time.time()
    
    async with httpx.AsyncClient(timeout=60.0) as client:
        tasks = []
        
        # 배치로 분할
        for i in range(0, len(orders), batch_size):
            batch = orders[i:i + batch_size]
            task = client.post(
                f"{BASE_URL}/kafka/batch/publish",
                json={
                    "messages": batch
                }
            )
            tasks.append(task)
        
        # 모든 배치 동시 실행
        responses = await asyncio.gather(*tasks)
        
        # 성공 확인
        total_sent = sum(
            r.json().get('messages_sent', 0) 
            for r in responses if r.status_code == 200
        )
    
    elapsed = time.time() - start_time
    
    print(f"⚡ 배치 전송 결과:")
    print(f"   총 처리 시간: {elapsed:.2f}초")
    print(f"   배치 크기: {batch_size}건")
    print(f"   배치 수: {len(responses)}회")
    print(f"   성공: {total_sent}/{len(orders)}건")
    print(f"   처리량: {len(orders)/elapsed:.2f} messages/sec")
    
    return elapsed

# 실행
print("🚀 배치 전송 시작 (100건 × 10회)...\n")
batch_time = await send_batch_kafka(orders, batch_size=100)
print(f"\n✅ 완료!")

## Step 3: Redis 캐싱 비교 — 3가지 방법의 차이

### ⚠️ 흔한 오개념: `asyncio.gather` ≠ Redis Pipeline

| 방법 | 구현 | Redis 측에서 본 RTT |
|------|------|-------------------|
| 개별 순차 HTTP | `for order: await client.post(...)` | 1000번 개별 왕복 |
| 동시 HTTP | `asyncio.gather(*tasks)` | 여전히 1000번 개별 Redis SET |
| **진짜 Redis Pipeline** | `aioredis.pipeline().execute()` | **1번의 TCP 패킷으로 전부** |

**`asyncio.gather`가 Redis Pipeline이 아닌 이유:**

```text
asyncio.gather (동시 HTTP):
  Client ──[HTTP POST /redis/cache/set]──→ FastAPI ──[SET key1]──→ Redis
  Client ──[HTTP POST /redis/cache/set]──→ FastAPI ──[SET key2]──→ Redis   (1000번 왕복)
  ...
  → HTTP 레벨 동시성일 뿐, Redis에는 여전히 명령 1개씩 도달

진짜 Redis Pipeline (aioredis):
  Client ──[SET key1, SET key2, ..., SET key1000]──→ Redis ──[결과 1000개]──→ Client
                  ↑ 하나의 TCP 소켓에 1000개 명령 묶음          ↑ 1번에 수신
  → Redis 서버가 한 번에 파싱 후 한 번에 처리 (메모리 버스 효율 극대화)
```

**예상 성능:**
- 순차 HTTP 1000건: ~수십 초 (network RTT × 1000)
- 동시 HTTP 1000건: ~1~3초 (concurrent, but HTTP overhead × 1000)
- Redis Pipeline: ~수십 ms (단 1번의 TCP 왕복)

In [ ]:
import redis.asyncio as _aioredis_bulk

async def cache_sequential_http(orders, sample=50):
    """1건씩 순서대로 전송 — 가장 느린 기준선"""
    start = time.time()
    async with httpx.AsyncClient(timeout=60.0) as c:
        for order in orders[:sample]:
            await c.post(f"{BASE_URL}/redis/cache/set",
                json={"key": f"order:{order['order_id']}", "value": order, "ttl": 3600})
    elapsed = time.time() - start
    per_req = elapsed / sample * 1000
    print(f"🐢 순차 HTTP ({sample}건): {elapsed:.2f}s | 요청당 {per_req:.1f}ms")
    return elapsed * (len(orders) / sample)  # 1000건 환산 추정

async def cache_concurrent_http(orders):
    """asyncio.gather로 동시에 HTTP 요청 — 빠르지만 Redis Pipeline이 아님"""
    start = time.time()
    async with httpx.AsyncClient(timeout=60.0) as c:
        tasks = [c.post(f"{BASE_URL}/redis/cache/set",
            json={"key": f"order:{o['order_id']}", "value": o, "ttl": 3600})
            for o in orders]
        responses = await asyncio.gather(*tasks)
        success = sum(1 for r in responses if r.status_code == 200)
    elapsed = time.time() - start
    print(f"⚡ 동시 HTTP ({len(orders)}건): {elapsed:.2f}s | {len(orders)/elapsed:.0f} ops/sec | 성공 {success}건")
    print("   ↑ HTTP 레벨 동시성 — Redis에는 여전히 명령 1개씩 도달")
    return elapsed

async def cache_real_pipeline(orders):
    """aioredis.pipeline() — 1000개 SET을 단 1번의 TCP 왕복으로 처리"""
    start = time.time()
    r = _aioredis_bulk.from_url("redis://localhost:6379")
    pipe = r.pipeline()
    for order in orders:
        pipe.set(f"order:{order['order_id']}", json.dumps(order), ex=3600)
    results = await pipe.execute()
    await r.aclose()
    elapsed = time.time() - start
    success = sum(1 for v in results if v)
    print(f"🚀 Redis Pipeline ({len(orders)}건): {elapsed:.2f}s | {len(orders)/elapsed:.0f} ops/sec | 성공 {success}건")
    print("   ↑ 진짜 Pipeline — 1번의 TCP 왕복으로 1000개 명령 처리")
    return elapsed

print("=== 방법 A: 순차 HTTP (50건 측정 → 1000건 추정) ===")
redis_seq_1000_est = await cache_sequential_http(orders, sample=50)
print(f"   1000건 환산 추정: {redis_seq_1000_est:.1f}s\n")

print("=== 방법 B: 동시 HTTP (asyncio.gather, 1000건) ===")
redis_individual_time = await cache_concurrent_http(orders)  # 기존 변수명 유지

print("\n=== 방법 C: 진짜 Redis Pipeline (aioredis, 1000건) ===")
redis_pipeline_time = await cache_real_pipeline(orders)

print(f"\n{'='*55}")
print(f"{'방법':<30} {'시간(s)':>8} {'속도향상':>10}")
print(f"{'-'*55}")
print(f"{'순차 HTTP (1000건 추정)':<30} {redis_seq_1000_est:>8.1f} {'1.0x (기준)':>10}")
print(f"{'동시 HTTP (asyncio.gather)':<30} {redis_individual_time:>8.2f} {redis_seq_1000_est/redis_individual_time:>9.1f}x")
print(f"{'Redis Pipeline (aioredis)':<30} {redis_pipeline_time:>8.2f} {redis_seq_1000_est/redis_pipeline_time:>9.1f}x")
print(f"{'='*55}")

## Step 4: 성능 비교 차트

pandas를 사용하여 벤치마크 결과를 정리하고 시각화합니다.

In [ ]:
# Step 4: 성능 비교 분석 및 시각화
import pandas as pd

# 벤치마크 결과 정리
benchmark_data = {
    '테스트': [
        'Kafka 개별 전송',
        'Kafka 배치 전송',
        'Redis 개별 캐싱',
        'Redis Pipeline 캐싱'
    ],
    '처리 시간 (초)': [
        individual_time,
        batch_time,
        redis_individual_time,
        redis_pipeline_time
    ],
    '처리량 (ops/sec)': [
        1000 / individual_time,
        1000 / batch_time,
        1000 / redis_individual_time,
        1000 / redis_pipeline_time
    ]
}

df = pd.DataFrame(benchmark_data)

# 속도 향상 배수 계산
df['속도 향상'] = [
    '1.0x (기준)',
    f'{individual_time/batch_time:.1f}x',
    '1.0x (기준)',
    f'{redis_individual_time/redis_pipeline_time:.1f}x'
]

print("\n📊 벤치마크 결과 요약:")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)

# 간단한 텍스트 기반 바 차트
print("\n📈 처리 시간 비교 (텍스트 차트):")
print("=" * 80)
max_time = max(benchmark_data['처리 시간 (초)'])
for test, time_val in zip(benchmark_data['테스트'], benchmark_data['처리 시간 (초)']):
    bar_length = int((time_val / max_time) * 50)
    bar = '█' * bar_length
    print(f"{test:25s} | {bar} {time_val:.2f}초")
print("=" * 80)

# matplotlib 사용 가능한 경우 시각화
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Kafka 비교
    kafka_data = df.iloc[:2]
    axes[0].bar(kafka_data['테스트'], kafka_data['처리 시간 (초)'], color=['#ff6b6b', '#51cf66'])
    axes[0].set_title('Kafka: 개별 vs 배치 전송', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('처리 시간 (초)', fontsize=12)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Redis 비교
    redis_data = df.iloc[2:]
    axes[1].bar(redis_data['테스트'], redis_data['처리 시간 (초)'], color=['#ff6b6b', '#51cf66'])
    axes[1].set_title('Redis: 개별 vs Pipeline 캐싱', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('처리 시간 (초)', fontsize=12)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ 시각화 차트 생성 완료!")
except ImportError:
    print("\n💡 matplotlib 미설치: 텍스트 차트만 표시됨")
except Exception as e:
    print(f"\n⚠️  차트 생성 실패: {e}")

## 결과 분석

### 1. Kafka 배치 전송의 효과

**관찰된 성능 향상:**
- 개별 전송 대비 **10~100배** 빠름
- 네트워크 왕복 횟수 감소: 1000회 → 10회

**원인:**
- HTTP 연결 재사용 (Keep-Alive)
- Kafka Producer의 배치 압축
- 네트워크 대역폭 효율적 활용
- I/O 대기 시간 최소화

### 2. Redis 3가지 방법 비교

| 방법 | Redis 관점 | 예상 시간 |
|------|----------|---------|
| 순차 HTTP | RTT × 1000번 개별 왕복 | ~수십 초 |
| 동시 HTTP (`asyncio.gather`) | 여전히 1000번 개별 SET | ~1~3초 |
| **진짜 Pipeline** (`aioredis`) | **1번의 TCP 왕복** | **~수십 ms** |

**핵심:** `asyncio.gather`는 HTTP 레벨 동시성이지, Redis Pipeline이 아닙니다.
Pipeline은 aioredis를 직접 사용해야만 가능합니다.

### 3. 실무 적용 시사점

**대용량 처리 시 필수 패턴:**
- Kafka: 배치 전송으로 RTT 횟수 자체를 줄인다
- Redis: 진짜 Pipeline (aioredis)으로 TCP 왕복을 1번으로 줄인다
- `asyncio.gather`는 HTTP 처리량을 높이지만 Redis RTT는 줄이지 못한다

**최적 배치 크기:**
- Kafka: 100~1000개 (메시지 크기에 따라)
- Redis Pipeline: 1000~10000개 (메모리 충분 시)

In [ ]:
# Step 5: 성능 향상 요약
print("\n" + "="*80)
print("🎯 성능 최적화 요약")
print("="*80)

# Kafka 속도 향상
kafka_speedup = individual_time / batch_time
print(f"\n📦 Kafka 배치 전송:")
print(f"   속도 향상: {kafka_speedup:.1f}배")
print(f"   시간 단축: {individual_time - batch_time:.2f}초 절약")
print(f"   효율성: {(1 - batch_time/individual_time)*100:.1f}% 개선")

if kafka_speedup > 50:
    print("   평가: ⭐⭐⭐ 탁월한 성능 향상!")
elif kafka_speedup > 20:
    print("   평가: ⭐⭐ 매우 좋은 성능 향상!")
elif kafka_speedup > 10:
    print("   평가: ⭐ 좋은 성능 향상!")

# Redis 속도 향상 (순차 HTTP 추정치 기준)
redis_concurrent_speedup = redis_seq_1000_est / redis_individual_time
redis_pipeline_speedup = redis_seq_1000_est / redis_pipeline_time
print(f"\n💾 Redis 성능 비교 (순차 HTTP 1000건 추정 기준):")
print(f"   동시 HTTP:    {redis_concurrent_speedup:.1f}배 빠름")
print(f"   Redis Pipeline: {redis_pipeline_speedup:.1f}배 빠름")
print(f"   Pipeline vs 동시HTTP: {redis_individual_time/redis_pipeline_time:.1f}배 빠름")

if redis_pipeline_speedup > 100:
    print("   평가: ⭐⭐⭐ Pipeline으로 100배 이상 향상!")
elif redis_pipeline_speedup > 20:
    print("   평가: ⭐⭐ Pipeline으로 큰 폭 향상!")
elif redis_pipeline_speedup > 5:
    print("   평가: ⭐ Pipeline 효과 확인!")

# 전체 요약
print(f"\n💡 핵심 인사이트:")
print(f"   asyncio.gather ≠ Redis Pipeline! HTTP 동시성과 TCP 배치는 다른 개념")
print(f"   진짜 Pipeline(aioredis): 1번의 TCP 왕복으로 1000개 명령 처리")
print(f"   Kafka: 배치 {kafka_speedup:.0f}배, Redis Pipeline: {redis_pipeline_speedup:.0f}배 이상 향상")

print("\n" + "="*80)

## 핵심 포인트: 배치 처리의 원리

### 1. 네트워크 효율성

**개별 순차 전송:**
```
Client → Server (메시지 1) → RTT
Client → Server (메시지 2) → RTT
...
총 시간 = RTT × N
```

**Kafka 배치 전송:**
```
Client → Server (메시지 1-100) → RTT
Client → Server (메시지 101-200) → RTT
...
총 시간 = RTT × (N / batch_size)
```

**Redis 진짜 Pipeline (aioredis):**
```
Client ──[SET key1, SET key2, ..., SET key1000]──→ Redis ──[결과 1000개]──→ Client
              ↑ 하나의 TCP 소켓, 1번의 왕복                 ↑ 1번에 수신
총 시간 = RTT × 1  (명령 수와 무관)
```

### 2. asyncio.gather vs Redis Pipeline — 완전히 다른 개념

```text
asyncio.gather (HTTP 동시성):
  Python  ──[POST /redis/cache/set]──→ FastAPI ──[SET key1]──→ Redis
  Python  ──[POST /redis/cache/set]──→ FastAPI ──[SET key2]──→ Redis  (1000개 병렬)
  ...
  ✅ HTTP 요청이 동시에 날아감
  ❌ Redis 서버에는 여전히 명령이 1개씩 개별 도달

aioredis.pipeline():
  Python ──[SET key1\r\nSET key2\r\n...SET key1000\r\n]──→ Redis
                      ↑ 하나의 TCP 패킷 or 소켓 버퍼에 묶음
  Redis ──[OK\r\nOK\r\n...OK\r\n]──────────────────────→ Python
  ✅ Redis가 한 번에 1000개 명령을 파싱·처리
  ✅ 네트워크 왕복 = 단 1번
```

### 3. Kafka/Redis 내부 최적화

**Kafka Producer 배치:**
- 압축 (compression): 배치 단위로 압축
- 버퍼링: 여러 메시지를 모아서 한 번에 전송
- 파티션 최적화: 동일 파티션 메시지 묶음

**Redis Pipeline (aioredis):**
- `r.pipeline()`으로 명령 큐 생성
- `pipe.set(...)` 은 즉시 실행이 아닌 큐에 추가
- `await pipe.execute()` 시점에 단 1번의 TCP 왕복으로 전부 전송
- Redis 서버가 한 번에 파싱 → 메모리 버스 효율 극대화

### 4. 실전 배치 크기 결정

**고려 요소:**
1. **메시지 크기**: 큰 메시지 → 작은 배치
2. **메모리**: 배치가 메모리에 적재되므로 제한 고려
3. **지연시간**: 배치가 클수록 대기 시간 증가
4. **처리량**: 배치가 클수록 처리량 증가

**권장 배치 크기:**
- Kafka: 100~1000개 (또는 1MB~10MB)
- Redis Pipeline: 1000~10000개
- 데이터베이스: 100~1000개

## 확장 아이디어

### 1. 압축 (Compression)

**Kafka 압축 알고리즘:**
```python
# Producer 설정
compression_type = 'snappy'  # or 'gzip', 'lz4', 'zstd'
```

**효과:**
- 네트워크 대역폭 70~90% 절감
- 저장 공간 절약
- CPU 사용량 증가 (트레이드오프)

### 2. 파티셔닝 전략 (Partitioning)

**효율적인 파티션 분배:**
```python
# 사용자 ID 기반 파티셔닝
partition_key = user_id  # 동일 사용자 주문은 같은 파티션
```

**이점:**
- 순서 보장 (파티션 내)
- 병렬 처리 가능
- 컨슈머 확장 용이

### 3. 배압 (Backpressure) 처리

**문제:**
- 프로듀서가 너무 빠르면 브로커/컨슈머 과부하

**해결책:**
```python
# 동적 배치 크기 조정
if queue_depth > threshold:
    batch_size = batch_size // 2  # 배치 크기 줄임
    delay = 0.1  # 지연 추가
```

### 4. 모니터링 지표

**측정 항목:**
- 처리 시간 (latency)
- 처리량 (throughput)
- 에러율 (error rate)
- 큐 깊이 (queue depth)
- CPU/메모리 사용률

### 5. 실전 최적화 예시

**쿠팡 규모 (가정):**
```
일일 주문: 10,000,000건
배치 크기: 1000건
배치 수: 10,000회
개별 전송 시간: 10,000,000 × 10ms = 27.8시간
배치 전송 시간: 10,000 × 10ms = 100초 = 1.7분

시간 절약: 27.8시간 → 1.7분 (약 1000배)
```

### 6. 추가 도전 과제

1. **동적 배치 크기:** 네트워크 상태에 따라 자동 조절
2. **실패 처리:** 부분 실패 시 재시도 전략
3. **순서 보장:** 순서가 중요한 메시지 처리
4. **멀티 리전:** 지리적으로 분산된 시스템에서의 배치 처리
5. **스트림 처리:** 실시간 배치 vs 마이크로 배치

In [ ]:
# Cleanup - 테스트 데이터 정리
print("🧹 정리 중...\n")

async def cleanup():
    """테스트 데이터 정리"""
    try:
        async with httpx.AsyncClient(timeout=30.0) as client:
            # Redis 캐시 정리: 각 order 키를 개별 삭제
            delete_tasks = [
                client.delete(f"{BASE_URL}/redis/cache/delete/order:{order['order_id']}")
                for order in orders[:100]  # 샘플 100건만 정리 (전체는 TTL로 자동 만료)
            ]
            responses = await asyncio.gather(*delete_tasks)
            deleted = sum(1 for r in responses if r.status_code == 200)
            print(f"✅ Redis 캐시 정리: {deleted}건 삭제 (나머지는 TTL 만료 대기)")
    except Exception as e:
        print(f"⚠️  정리 중 오류: {e}")
    
    print("\n💡 참고: Kafka 메시지는 보존 정책에 따라 자동 정리됩니다.")
    print("💡 벤치마크 데이터 파일은 유지됩니다: ../data/mock/bulk_orders.json")

await cleanup()
print("\n✅ 정리 완료!")

## 네비게이션

**이전 과제:**
- [← Challenge 13: 분산 추적](13_challenge_distributed_tracing.ipynb)

**다음 과제:**
- [→ Challenge 15: 고급 패턴](15_challenge_advanced.ipynb)

**관련 노트북:**
- [04: Kafka 배치 작업](04_kafka_batch_operations.ipynb)
- [05: Redis 캐싱](05_redis_caching.ipynb)

---

## 학습 요약

이 챌린지에서 배운 내용:

1. ✅ **배치 처리의 중요성**: 개별 처리 대비 수십~수백배 성능 향상
2. ✅ **네트워크 RTT**: 대용량 처리의 가장 큰 병목 지점
3. ✅ **Kafka 배치 전송**: Producer 배치 최적화
4. ✅ **Redis Pipeline**: 네트워크 왕복 최소화
5. ✅ **성능 벤치마킹**: 측정 방법론 및 지표 분석
6. ✅ **실무 최적화**: 배치 크기 결정 및 트레이드오프

**실전 적용:**
- 항상 배치 처리를 우선 고려
- 네트워크 왕복 횟수 최소화
- 배치 크기는 메모리와 지연시간 고려
- 압축 및 파티셔닝으로 추가 최적화

---

**Challenge 14 완료!** 대용량 데이터 처리 최적화 마스터! 🎉